# CMM synthetic validation

CMM uses flexmix (mixture of linear regressions), which assumes Gaussian conditional distributions.
Real TB data has binary mutation columns, handled via a Gaussian noise workaround.

This notebook tests whether CMM reliably recovers causal edges under two conditions:
- **binary → continuous** (mutation → MIC): the edge type we care about most
- **binary → binary** (mutation → mutation): the edge type dominating the real bootstrap results
```
Mixture structure: latent Z creates two subpopulations with different causal mechanisms.

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../external/cmm')

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
from src_tb.data.synthetic import generate_synthetic, visualize_synthetic_dag
from src_tb.causal_recovery.evaluation import eval_recovery
from src_tb.causal_recovery.cmm_utils import bootstrap_cmm, get_stable_edges, build_stable_bn
import pyagrum.lib.notebook as gnb

N_MUTATIONS = 8
N_SEEDS = 5
N_BOOTSTRAP = 10
THRESHOLD = 0.5

In [ ]:
# show the true DAG for the chosen n_mutations
_, features_synth, true_edges, binary_indices_synth = generate_synthetic(N_MUTATIONS)

n_quarter = N_MUTATIONS // 4
true_direct     = {(s, t) for s, t in true_edges if t == 'Y' and int(s.split('_')[1]) < n_quarter}
true_bin_to_bin = {(s, t) for s, t in true_edges if t != 'Y'}
true_chain_cont = {(s, t) for s, t in true_edges if t == 'Y' and int(s.split('_')[1]) >= n_quarter}

print(f"Direct   (mut -> Y):          {true_direct}")
print(f"Binary to binary (mut -> mut):{true_bin_to_bin}")
print(f"Chain cont (mut -> Y via mut):{true_chain_cont}")
visualize_synthetic_dag(true_edges, features_synth, binary_indices_synth)

In [ ]:
records = []
for seed in range(N_SEEDS):
    X, features_synth, true_edges, binary_indices_synth = generate_synthetic(N_MUTATIONS, seed=seed)
    cmm_list = bootstrap_cmm(X, set(), binary_indices_synth, n_runs=N_BOOTSTRAP)
    stable = get_stable_edges(cmm_list, features_synth, threshold=THRESHOLD)
    stable_set = set(zip(stable['source'], stable['target']))

    pr_d,  re_d,  f1_d  = eval_recovery(stable_set, true_direct)
    pr_bb, re_bb, f1_bb = eval_recovery(stable_set, true_bin_to_bin)
    pr_cc, re_cc, f1_cc = eval_recovery(stable_set, true_chain_cont)
    records.append({
        'seed': seed, 'stable_edges': stable_set,
        'direct P': pr_d,   'direct R': re_d,   'direct F1': f1_d,
        'bin->bin P': pr_bb, 'bin->bin R': re_bb, 'bin->bin F1': f1_bb,
        'chain P': pr_cc,   'chain R': re_cc,   'chain F1': f1_cc,
    })
    print(f"seed {seed} | {stable_set}")
    print(f"  direct    P={pr_d}  R={re_d}  F1={f1_d}")
    print(f"  bin->bin  P={pr_bb} R={re_bb} F1={f1_bb}")
    print(f"  chain     P={pr_cc} R={re_cc} F1={f1_cc}")

df = pd.DataFrame(records)

In [ ]:
cols = ['direct P', 'direct R', 'direct F1',
        'bin->bin P', 'bin->bin R', 'bin->bin F1',
        'chain P', 'chain R', 'chain F1']
print("=== Mean across seeds ===")
print(df[cols].mean().round(3).to_string())

In [ ]:
X_last, features_synth, true_edges, binary_indices_synth = generate_synthetic(N_MUTATIONS, seed=N_SEEDS - 1)
cmm_list_last = bootstrap_cmm(X_last, set(), binary_indices_synth, n_runs=N_BOOTSTRAP)
bn = build_stable_bn(cmm_list_last, features_synth, threshold=THRESHOLD)
gnb.showBN(bn, size="20")